# Week 1 Project: Build a Coding Agent That Fixes Failing Tests

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/new-programs-ik/Applied-Agentic-AI-for-SWEs/blob/main/Week%201/project/Fix_Failing_Tests_Agent.ipynb)

## The problem

Every engineering team knows the routine: CI goes red, someone opens the test output, reads the traceback, opens three files, finds the real cause (often in a different file from the failing test), makes a fix, runs the tests again, and repeats until everything is green.

Tools like GitHub Copilot agent mode, Claude Code and Cursor now do this loop on their own. In this project you will build a small version of one **from scratch**, with nothing but an LLM API and about 150 lines of Python.

## What you will build

An agent that is given a small Python repository with a failing test suite and the goal: **make all the tests pass.** It has four tools:

| Tool | What it does |
|---|---|
| `list_files()` | See what is in the repository |
| `read_file(path)` | Read a source or test file |
| `write_file(path, content)` | Save a fixed version of a source file |
| `run_tests()` | Run `pytest` and see what passes and fails |

Nobody tells the agent which files to open, which bug to fix first, or how many attempts it gets. It decides, acts, looks at the test results, and decides again: the **Think → Act → Observe** loop from class.

## What you will learn

1. How a real agent loop works, end to end, with no framework hiding it.
2. The difference between a **workflow** (you hard-code the steps) and an **agent** (the model picks the steps), by building both on the same problem.
3. Why tools that give **real feedback** (here: test results) are the best defence against hallucination.
4. Basic guardrails every agent needs: sandboxed file access, protected files, a step limit, and checking the result yourself instead of trusting the agent's "done".

## Why this is the Week 1 project

This project uses **only** what this week's class covers, and practises every part of it:

| Concept from this week's class | Where it shows up in this notebook |
|---|---|
| Calling LLMs programmatically | One OpenAI-SDK client, routed through OpenRouter, exactly as in the "Calling LLMs" notebook |
| Prompt engineering: context, structure, constraints | The agent's system prompt uses the `## Role / ## Task / ## Rules / ## Output` structure from the prompt engineering slide |
| Limitations of LLMs: no actions, hallucinations | The model can't run code or see your files on its own, and it can be confidently wrong. Tools give it actions; `run_tests` catches wrong fixes |
| Agent = (Prompt + Tools + Memory) × LLM | Prompt = system prompt, Tools = the four functions, Memory = the message history that carries every earlier attempt |
| Agents and tools: retrieve data, execute actions, run code | `read_file` retrieves, `write_file` acts, `run_tests` executes code |
| Workflows vs. autonomous agents | Part 1 builds a fixed pipeline; Part 2 builds the agent. You compare them on the same bugs |
| The agent loop / ReAct (Think → Act → Observe) | The `run_agent` loop prints each step so you can read the agent's trace |
| Planning and reflection | When a fix fails, the agent has to read the new error and revise its own change |

It is also sized for Week 1:

- **No frameworks.** No LangChain, LangGraph, vector databases or deployment. Those come in later weeks, and they will make more sense once you have written the loop yourself.
- **Runs in Colab with nothing to install.** `openai` and `pytest` are already in Colab. The only secret you need is `OPENROUTER_APIKEY`.
- **Cheap.** A full run with the default model costs about one to two cents.
- **Real, not simulated.** The tools act on real files and run real tests. If the agent's fix is wrong, the tests say so, and the agent has to deal with it.

# Setup

In [ ]:
%pip install -q openai pytest

In [ ]:
import difflib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

from openai import OpenAI

### Connect to the model

As in the other Week 1 notebooks, we use the official `openai` SDK pointed at OpenRouter. Add `OPENROUTER_APIKEY` in Colab Secrets (the key icon in the left sidebar) and allow this notebook to read it.

In [ ]:
try:
    from google.colab import userdata
    api_key = userdata.get("OPENROUTER_APIKEY")
except ImportError:  # running outside Colab: read it from an environment variable instead
    api_key = os.environ.get("OPENROUTER_APIKEY")

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

# Any OpenRouter model that supports tool calling works here, for example
# "anthropic/claude-haiku-4.5", "google/gemini-2.5-flash" or "openai/gpt-4o".
MODEL = "openai/gpt-4.1-mini"

# Step 1: Create the buggy repository

The cell below writes a tiny e-commerce helper library called `shopkit` into a folder next to this notebook. It has three source modules and a test suite:

```
shopkit_repo/
├── shopkit/
│   ├── pagination.py   # split product lists into pages
│   ├── pricing.py      # discount maths
│   ├── cart.py         # shopping cart totals (uses pricing.py)
│   └── text.py         # turn product names into URL slugs
└── tests/
    ├── test_pagination.py
    ├── test_cart.py
    └── test_text.py
```

Some bugs have been planted. Some are easy to spot. At least one is in a different file from the test that fails, and at least one has an obvious fix that is only partly right. The answer key is at the end of the notebook. Try not to read it before the agent runs.

You can run `create_repo()` again at any time to reset the repository to its buggy starting state.

In [ ]:
REPO_DIR = Path("shopkit_repo").resolve()

ORIGINAL_FILES = {
    "shopkit/__init__.py": '''''',

    "shopkit/pagination.py": '''"""Split long product lists into pages for display."""
import math


def paginate(items, page, page_size):
    """Return the items on the given page. Pages are numbered from 1."""
    start = page * page_size
    return items[start:start + page_size]


def total_pages(item_count, page_size):
    """Return how many pages are needed to show item_count items."""
    return item_count // page_size
''',

    "shopkit/pricing.py": '''"""Price calculations shared by the cart and checkout."""


def apply_discount(price, percent):
    """Return the price after a percentage discount.

    Example: apply_discount(200, 10) -> 180.0
    """
    return round(price * percent / 100, 2)
''',

    "shopkit/cart.py": '''"""A minimal shopping cart."""
from shopkit.pricing import apply_discount


class Cart:
    def __init__(self):
        self.items = []

    def add_item(self, name, unit_price, quantity=1):
        self.items.append({"name": name, "unit_price": unit_price, "quantity": quantity})

    def subtotal(self):
        return round(sum(i["unit_price"] * i["quantity"] for i in self.items), 2)

    def total(self, discount_percent=0):
        """Subtotal after an optional whole-cart percentage discount."""
        return apply_discount(self.subtotal(), discount_percent)
''',

    "shopkit/text.py": '''"""Text helpers for product pages."""


def slugify(name):
    """Turn a product name into a URL slug.

    Lowercase ASCII letters and digits only, words joined by single hyphens,
    accents removed, apostrophes dropped, no leading or trailing hyphens.
    """
    return name.lower().replace(" ", "-")
''',

    "tests/test_pagination.py": '''from shopkit.pagination import paginate, total_pages

ITEMS = list(range(1, 26))  # 25 products


def test_first_page():
    assert paginate(ITEMS, page=1, page_size=10) == list(range(1, 11))


def test_last_partial_page():
    assert paginate(ITEMS, page=3, page_size=10) == [21, 22, 23, 24, 25]


def test_total_pages_rounds_up():
    assert total_pages(25, 10) == 3


def test_total_pages_exact():
    assert total_pages(20, 10) == 2
''',

    "tests/test_cart.py": '''from shopkit.cart import Cart


def make_cart():
    cart = Cart()
    cart.add_item("T-shirt", 20.00, quantity=2)
    cart.add_item("Cap", 15.50)
    return cart


def test_subtotal():
    assert make_cart().subtotal() == 55.50


def test_total_without_discount():
    assert make_cart().total() == 55.50


def test_total_with_discount():
    assert make_cart().total(discount_percent=10) == 49.95
''',

    "tests/test_text.py": '''from shopkit.text import slugify


def test_simple_name():
    assert slugify("Blue Shirt") == "blue-shirt"


def test_extra_whitespace():
    assert slugify("  Red   Hat  ") == "red-hat"


def test_punctuation_and_apostrophes():
    assert slugify("Men's T-Shirt (XL)") == "mens-t-shirt-xl"


def test_accents_are_removed():
    assert slugify("Café Crème Mug") == "cafe-creme-mug"
''',
}


def create_repo():
    """(Re)create the buggy repository from ORIGINAL_FILES."""
    for rel_path, content in ORIGINAL_FILES.items():
        path = REPO_DIR / rel_path
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content, encoding="utf-8")
    print(f"Repository ready at {REPO_DIR}")


create_repo()

### See the failures yourself

Before handing anything to an LLM, run the tests the way a developer would. This output is exactly what the agent will see when it calls `run_tests`.

In [ ]:
def run_pytest():
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "--tb=short", "-p", "no:cacheprovider"],
        cwd=REPO_DIR, capture_output=True, text=True, timeout=60,
    )
    return result.returncode == 0, result.stdout + result.stderr


passed, output = run_pytest()
print(output)

# Step 2: Build the tools

Tools are ordinary Python functions. Three design rules make them safe and useful for an agent:

1. **Return strings, never raise.** If the agent asks for a file that doesn't exist, it should get a clear error message it can react to, not crash the loop. Errors are observations too.
2. **Stay inside the sandbox.** Every path is resolved and checked against `REPO_DIR`, so the agent can't read or write `../../anything`.
3. **Protect what defines "done".** The agent may not edit anything in `tests/`. Otherwise the easiest way to make tests pass is to change the tests, and a goal-driven agent will sometimes find that shortcut.

We also cut long outputs short, because everything a tool returns is sent back to the model on every later step, and that costs tokens.

In [ ]:
MAX_TOOL_OUTPUT = 4000  # characters


def _resolve(path):
    """Turn a repo-relative path into an absolute one, refusing anything outside the repo."""
    full = (REPO_DIR / path).resolve()
    if not full.is_relative_to(REPO_DIR):
        raise ValueError(f"'{path}' is outside the repository")
    return full


def list_files():
    files = [
        str(p.relative_to(REPO_DIR)).replace("\\", "/")
        for p in sorted(REPO_DIR.rglob("*.py"))
        if "__pycache__" not in p.parts
    ]
    return "\n".join(files)


def read_file(path):
    try:
        full = _resolve(path)
        if not full.is_file():
            return f"ERROR: file not found: {path}"
        return full.read_text(encoding="utf-8")[:MAX_TOOL_OUTPUT]
    except ValueError as e:
        return f"ERROR: {e}"


def write_file(path, content):
    try:
        full = _resolve(path)
    except ValueError as e:
        return f"ERROR: {e}"
    if full.is_relative_to(REPO_DIR / "tests"):
        return "ERROR: files in tests/ are read-only. Fix the source code, not the tests."
    if full.suffix != ".py":
        return "ERROR: only .py files can be written."
    full.parent.mkdir(parents=True, exist_ok=True)
    full.write_text(content, encoding="utf-8")
    return f"Wrote {len(content)} characters to {path}"


def run_tests():
    passed, output = run_pytest()
    status = "ALL TESTS PASSED" if passed else "SOME TESTS FAILED"
    return f"{status}\n\n{output[-MAX_TOOL_OUTPUT:]}"


AVAILABLE_TOOLS = {
    "list_files": list_files,
    "read_file": read_file,
    "write_file": write_file,
    "run_tests": run_tests,
}

Try the guardrails before the agent does:

In [ ]:
print(read_file("../secrets.txt"))
print(write_file("tests/test_cart.py", "def test_ok(): pass"))
print(read_file("shopkit/missing.py"))

### Describe the tools to the model

The model never sees the Python functions, only these JSON schemas. The descriptions matter: they are how the model decides **when** to use each tool. Notice that `write_file` says it replaces the whole file, so the model knows to send the complete file and not just the changed lines.

In [ ]:
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List all Python files in the repository, as paths relative to the repo root.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the full contents of a file in the repository.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Path relative to the repo root, e.g. 'shopkit/cart.py'"},
                },
                "required": ["path"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": (
                "Replace the ENTIRE contents of a source file with new content. "
                "Always send the complete file, not just the changed lines. Files in tests/ are read-only."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Path relative to the repo root, e.g. 'shopkit/cart.py'"},
                    "content": {"type": "string", "description": "The complete new contents of the file."},
                },
                "required": ["path", "content"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "run_tests",
            "description": "Run the full pytest suite and return a pass/fail summary with any failure details.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
]

# Part 1: The workflow version (the developer decides the steps)

Before building the agent, let's solve the problem the "traditional" way: a **fixed pipeline** whose steps are hard-coded by us.

1. Run the tests once.
2. For each failing test file `tests/test_X.py`, assume the bug is in `shopkit/X.py`.
3. Send that test file, that source file and the failures to the LLM in **one** call, and ask for the fixed file.
4. Save it. Run the tests one last time and report.

This is a perfectly reasonable design, and for many well-defined jobs a workflow like this is the right choice: it is predictable, and it makes a known, fixed number of LLM calls. But look at the assumptions we baked in: the bug is always in the file with the matching name, and one attempt is enough. Watch what happens when the real world doesn't follow them.

In [ ]:
def extract_code(text):
    """Pull the first ```python block out of a model reply (or return the reply as-is)."""
    match = re.search(r"```(?:python)?\n(.*?)```", text, re.S)
    return match.group(1) if match else text


def run_workflow():
    _, output = run_pytest()
    failing_test_files = sorted(set(re.findall(r"tests/test_\w+\.py", output)))
    print(f"Failing test files: {failing_test_files}")
    cost = 0.0

    for test_file in failing_test_files:
        module = "shopkit/" + test_file.removeprefix("tests/test_")  # tests/test_cart.py -> shopkit/cart.py
        prompt = (
            f"These tests are failing.\n\n## Test file: {test_file}\n{read_file(test_file)}\n\n"
            f"## Source file: {module}\n{read_file(module)}\n\n## pytest output\n{output}\n\n"
            f"Return the complete fixed version of {module} in a single ```python code block."
        )
        reply = client.chat.completions.create(
            model=MODEL, messages=[{"role": "user", "content": prompt}]
        )
        cost += (reply.usage.model_extra or {}).get("cost") or 0
        print(f"  LLM call -> {write_file(module, extract_code(reply.choices[0].message.content))}")

    passed, output = run_pytest()
    print(f"\n--- {len(failing_test_files)} model calls, cost ${cost:.4f} ---")
    print("Result:", "ALL TESTS PASSED" if passed else "SOME TESTS FAILED")
    print(output)


create_repo()
run_workflow()

### What usually happens

Results vary between runs and models, but you will usually see one of these:

- **Some tests still fail, and the pipeline just stops.** It has no step for "look at the new error and try again", because we never wrote one.
- **The cart tests "pass" because of a workaround in `cart.py`.** The real bug is in `pricing.py`, which the pipeline never showed the model. A model that only sees `cart.py` may skip `apply_discount` and do the discount maths inline. The tests turn green, but the bug is still there for every other caller of `apply_discount`. Run the diff cell below to check.

Either way, the weakness is the same: **we decided the steps in advance, and the problem didn't fit them.**

In [ ]:
def show_diff():
    """Show what changed compared with the original buggy repo, like reviewing a pull request."""
    for rel_path, original in ORIGINAL_FILES.items():
        current = (REPO_DIR / rel_path).read_text(encoding="utf-8")
        if current != original:
            print("".join(difflib.unified_diff(
                original.splitlines(keepends=True), current.splitlines(keepends=True),
                fromfile=f"a/{rel_path}", tofile=f"b/{rel_path}",
            )))


show_diff()

# Part 2: The agent version (the model decides the steps)

Now we give the model the goal, the tools and a loop, and let it decide everything else.

### The system prompt

The prompt follows the structure from the prompt engineering slide: **Role**, **Task**, **Rules** (constraints) and **Output**. Compare it with the workflow above: it says *what* done looks like and *what is not allowed*, but not *which files to open in which order*. That freedom is what makes this an agent.

In [ ]:
SYSTEM_PROMPT = """## Role
You are a careful senior Python engineer fixing a failing test suite.

## Task
Make every test in the repository pass by fixing bugs in the source code.

## Rules
- Run the tests first to see what is failing.
- Read the relevant source files before changing anything. A failing test can be caused by a bug in a different file from the one it tests: follow the imports.
- Fix the root cause, where the bug actually lives. Do not work around a bug in one module by changing another.
- Never edit files in tests/. The tests define the correct behaviour.
- Keep changes small. Do not rewrite code that already works.
- After every change, run the tests again. If something still fails, read the new error and revise your fix.
- Stop when all tests pass.

## Output
When you are finished, reply with a short summary: for each bug, the file, the root cause, and the fix."""

### The agent loop

This is the whole agent. Each pass through the `for` loop is one **Think → Act → Observe** cycle:

- **Think:** send the full message history to the model. It either asks for tools or gives its final answer.
- **Act:** run every tool the model asked for.
- **Observe:** add each tool result to the history, so the model sees it on the next pass.

Two details are worth noticing:

- **`max_steps`** is a hard stop. Agents can get stuck repeating themselves, and every step costs money. Production agents always have a budget like this.
- **Memory is just the `messages` list.** The model has no memory of its own. It "remembers" its earlier failed fix only because we resend the whole conversation every step. This is also why the cost of each step grows as the run goes on.

In [ ]:
def run_agent(goal, max_steps=20):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": goal},
    ]
    totals = {"steps": 0, "tool_calls": 0, "input_tokens": 0, "output_tokens": 0, "cost": 0.0}

    for step in range(1, max_steps + 1):
        # THINK: the model looks at everything so far and decides what to do next
        response = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS_SCHEMA, tool_choice="auto",
        )
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))

        totals["steps"] = step
        totals["input_tokens"] += response.usage.prompt_tokens
        totals["output_tokens"] += response.usage.completion_tokens
        totals["cost"] += (response.usage.model_extra or {}).get("cost") or 0

        if message.content:
            print(f"\n[Step {step}] THINK: {message.content.strip()}")

        if not message.tool_calls:  # no more actions requested: the agent thinks it is done
            break

        for tool_call in message.tool_calls:
            # ACT: run the tool the model asked for
            name = tool_call.function.name
            args = {}
            try:
                args = json.loads(tool_call.function.arguments or "{}")
                result = AVAILABLE_TOOLS[name](**args) if name in AVAILABLE_TOOLS else f"ERROR: unknown tool {name}"
            except (json.JSONDecodeError, TypeError) as e:
                result = f"ERROR: bad arguments for {name}: {e}"
            totals["tool_calls"] += 1

            shown_args = {k: (v if k != "content" else f"<{len(v)} chars>") for k, v in args.items()} if isinstance(args, dict) else args
            print(f"[Step {step}] ACT: {name}({shown_args})")
            print(f"[Step {step}] OBSERVE: {result.splitlines()[0] if result else '(empty)'}")

            # OBSERVE: add the result to the history so the model sees it on the next step
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
    else:
        print(f"\nStopped: reached max_steps={max_steps} before the agent finished.")

    print(
        f"\n--- {totals['steps']} model calls, {totals['tool_calls']} tool calls, "
        f"{totals['input_tokens']:,} input / {totals['output_tokens']:,} output tokens, "
        f"cost ${totals['cost']:.4f} ---"
    )
    return messages

### Run it

We reset the repository first, so the agent starts from the same buggy code the workflow did.

In [ ]:
create_repo()
history = run_agent("The test suite in this repository is failing. Please fix it.")

# Step 3: Don't trust the agent. Verify.

The agent *says* it is done. An engineer would never merge a pull request just because the author says it works, so we check for ourselves:

1. Run the tests independently of the agent.
2. Review the diff, the same way you would review a teammate's pull request. Did it fix the root cause? Did it touch anything it shouldn't have?

In [ ]:
passed, output = run_pytest()
print("VERIFIED: all tests pass" if passed else "NOT DONE: tests still fail")
print(output)
show_diff()

### Reading the trace

Scroll back up through the agent's output and look for these moments. They are the whole point of the agent loop:

- **Exploring:** the agent reads files it was never told about, for example opening `pricing.py` because `cart.py` imports it.
- **Acting on evidence:** each `write_file` follows a `read_file` and a test failure, not a guess.
- **Reflection:** a `run_tests` result shows a fix that only partly worked, and the next step changes that fix. (Strong models sometimes get every fix right the first time. Try a smaller model, or run it a few times, to see this happen.)
- **Stopping:** the agent stops calling tools once `run_tests` reports success.

Then compare with Part 1:

| | Workflow (Part 1) | Agent (Part 2) |
|---|---|---|
| Who decides the next step? | Our code | The model |
| Number of LLM calls | Fixed: one per failing test file | Unknown in advance: as many as it needs (up to `max_steps`) |
| Finds a bug in an unexpected file | Only if we predicted it | Yes, by following imports |
| Recovers from a fix that didn't work | No | Yes: it reads the new error and tries again |
| Predictability and cost | High / low | Lower / higher |

Neither is "better" in general. As the class slide says, most production systems are **hybrids**: a fixed workflow for the overall stages, with an agent given freedom inside a bounded step. A CI bot might always run in the order "reproduce → agent fixes → tests verify → human reviews", while the "agent fixes" stage is open-ended.

# Exercises

Pick at least two. Each one builds on a concept from this week.

1. **Add a bug of your own.** Edit `ORIGINAL_FILES` to plant a new bug (and a test that catches it), then rerun the agent. Can you write a bug the agent can't fix in 20 steps? Why does it get stuck?
2. **Add a `search_code(pattern)` tool** that returns every line matching a regex, with file names and line numbers. Does the agent use it instead of reading whole files? Does the token count go down?
3. **Try to break the guardrails.** Change the goal to "Make the tests pass by any means necessary." Does the agent try to edit `tests/`? What does it do when the tool refuses?
4. **Plan first.** Change the system prompt so the agent must write a numbered plan before its first tool call. Does planning reduce the number of steps?
5. **Compare models.** Run the agent with three different values of `MODEL`. Record steps, tool calls, cost and whether the fix was correct. Which model would you pick for a CI bot, and why?
6. **Smarter edits.** Replace `write_file` with `replace_in_file(path, old, new)`, which changes only an exact snippet. Why do real coding agents prefer this? (Hint: think about tokens and accidental damage to the rest of the file.)

# Ship it to your portfolio

As with every agent in this course: move the code into a small Python project, write a README (problem, how the agent works, tools and guardrails, sample trace, limitations, next steps), and publish it to your GitHub.

---

# Answer key (instructors, or after you have run the agent)

| # | Bug | Failing tests | Root cause | Fix | What it tests in the agent |
|---|---|---|---|---|---|
| 1 | `paginate` uses `page * page_size` with 1-indexed pages | `test_first_page`, `test_last_partial_page` | Off-by-one in `pagination.py` | `start = (page - 1) * page_size` | Basic read → fix → verify |
| 2 | `total_pages` rounds down | `test_total_pages_rounds_up` | Floor division in `pagination.py` | `math.ceil(item_count / page_size)` (`math` is already imported) | A second bug in the same file |
| 3 | `apply_discount` returns the discount amount, not the discounted price | All of `test_cart.py` except `test_subtotal` | The bug is in `pricing.py`, but only `test_cart.py` fails | `round(price * (100 - percent) / 100, 2)` | Following imports to the real cause; not working around it in `cart.py` |
| 4 | `slugify` only replaces single spaces | 3 tests in `test_text.py` | Naive implementation in `text.py` | Normalise accents with `unicodedata`, drop apostrophes, replace runs of other characters with `-`, strip `-` from the ends | Reflection: the obvious regex fix `re.sub(r"[^a-z0-9]+", "-", ...)` still fails the apostrophe and accent tests |

A reference fix for bug 4:

```python
import re
import unicodedata


def slugify(name):
    ascii_name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    ascii_name = ascii_name.lower().replace("'", "")
    return re.sub(r"[^a-z0-9]+", "-", ascii_name).strip("-")
```